## Строим паплайн (пере)обучения модели на новых данных

Здесь загружаем весь датасет, совмещаем staging-словарь с основным main-словарем, переразмечаем датасет по обновленному словарю, предобрабатываем текстовые поля и запускаем переобучение модели на фиксированных параметрах.

In [1]:
#%pip install transformers==4.54.0 nlpaug catboost torch==2.6.0 spacy==3.6.0 --user

In [2]:
import pandas as pd
import numpy as np
import random
import os
import re
import requests
import sys, subprocess, importlib.util

import site
site.addsitedir(site.getusersitepackages())

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA

# ⏳ прогресс-бары
from tqdm import tqdm

# 🧠 обработка текста и NLP

import spacy
from spacy.util import is_package

MODEL_WHL = (
        "https://github.com/explosion/spacy-models/releases/download/"
        "ru_core_news_sm-3.6.0/ru_core_news_sm-3.6.0-py3-none-any.whl"
    )

if not is_package("ru_core_news_sm"):
    try:
        # для локальной установки
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "ru-core-news-sm==3.6.0"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    except subprocess.CalledProcessError:
        # для запуске в jobs
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-cache-dir",
                MODEL_WHL,
            ],
            stdout=subprocess.DEVNULL,
        )


import transformers
from transformers import AutoModel, AutoTokenizer

import nlpaug
import nlpaug.augmenter.char as nac
    
# 🤖 pyTorch
import torch

# загрузка моделей и функци для предобработки данных
from sklearn.model_selection import StratifiedKFold, GridSearchCV

import joblib

from catboost import CatBoostClassifier,utils
    
    
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

2026-02-08 21:16:23.431733: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-08 21:16:24.657205: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-08 21:16:28.157749: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
import spacy
print(spacy.__version__)   # 3.6.0

3.6.1


In [4]:
import torch
print(torch.__version__)

import transformers
print(transformers.__version__)

2.6.0+cu118
4.57.0


In [5]:
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [6]:
# создаем функцию очистки текста
def _clean_single_text(text):
    return re.sub(r"[^\w\s]", " ", text.lower())

# создаем функцию предобработки текста
def preprocess_texts_optimized(texts, nlp_model_name,
                               batch_size_cpu=256,
                               num_processes_for_cleaning=-1,
                               num_processes_for_spacy_cpu=-1):
    
    print(f"🔍 Запуск предобработки для {len(texts)} текстов...")
    
    # предварительная очистка текстов
    cleaned_texts = [_clean_single_text(text) for text in tqdm(texts, desc="Очистка")]

    # spaCy и лемматизация
    nlp = None
    processed_lemmas = []
    
    # загрузка NLP модели
    print(f"🔍 Загрузка spaCy модели: '{nlp_model_name}'.")
    nlp = spacy.load(nlp_model_name)

    # используем n_process для параллелизации
    if num_processes_for_spacy_cpu == -1:
        cpu_count = os.cpu_count()
        num_processes_for_spacy_cpu = max(1, cpu_count - 1)
    
    print(f"🔍 Лемматизация будет выполнена в {num_processes_for_spacy_cpu} потоках.")
    
    for doc in tqdm(nlp.pipe(cleaned_texts, batch_size=batch_size_cpu, n_process=num_processes_for_spacy_cpu), total=len(cleaned_texts), desc="Лемматизация (CPU)"):
        lemmas = [token.lemma_ for token in doc]
        processed_lemmas.append(" ".join(lemmas))
    
    print(f"🔍 Предобработка завершена. Обработано {len(processed_lemmas)} текстов.")
    return processed_lemmas

# создаем функцию для получения усредненного эмбеддинга текста
def get_embeddings_batch(texts, tokenizer, model, device, batch_size=64):
    texts = list(texts)
    embeddings = []

    print(f"🔍 Начало генерации эмбеддингов для {len(texts)} текстов на устройстве '{device}'.")
    for i in tqdm(range(0, len(texts), batch_size), desc="Генерация эмбеддингов"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        # Переносим каждый тензор на устройство
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # берем attention mask (1 — реальные токены, 0 — паддинг)
        mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size())
        masked_embeddings = outputs.last_hidden_state * mask

        # считаем среднее только по непаддинговым токенам
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts

        embeddings.extend(mean_pooled.cpu().numpy())
    
    print(f"🔍 Генерация эмбеддингов завершена")
    
    return embeddings

# функция предобработки входного датасета
def prepare_data(data, is_train, scaler=None, ipca=None, 
                                 scaler_art=None, ipca_art=None,
                                 scaler_pro=None, ipca_pro=None,
                                 scaler_cou=None, ipca_cou=None):
    
    data = data.drop(
            [
                "robot_id",
                "accounts__id",
                "articles__id",
                "articles__user_id",
                "projects__id",
                "projects__user_id",
                "counterparties__id",
                "counterparties__user_id",
                "robots__user_id",
                "article_alternative_names__user_id",
            ],
            axis=1,
        )

    # поправим типы данных и заполним пропуски метками missing (для текстовых значений категорий) и 0 для пропущенных ID
    data[
        [
            "articles__parent_id",
            "projects__parent_id",
            "counterparties__parent_id",
            "robots__id",
        ]
    ] = (
        data[
            [
                "articles__parent_id",
                "projects__parent_id",
                "counterparties__parent_id",
                "robots__id",
            ]
        ]
        .fillna(0)
        .astype("int64")
    )

    data["purpose"] = data["purpose"].fillna("missing")
    data["articles__name"] = data["articles__name"].fillna("missing")
    data["projects__name"] = data["projects__name"].fillna("missing")
    data["counterparties__name"] = data["counterparties__name"].fillna("missing")

    # переименуем и поправим тип столбца с фондами
    data = data.rename(columns={"accounts__user_id": "user_id"})
    data["user_id"] = data["user_id"].fillna(0).astype("int64")

    # кодируем текстовые поля
    # сначала очищаем и лемматизируем тексты
    data["clean_purpose"] = preprocess_texts_optimized(texts=data["purpose"],nlp_model_name="ru_core_news_sm")

    # грузим модели 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
    model = AutoModel.from_pretrained("DeepPavlov/rubert-base-cased")
    model = model.to(device)
    model.eval()
    
    # и запускаем генерацию эмбеддингов в назначении платежа
    data["purpose_emb"] = get_embeddings_batch(data["clean_purpose"], tokenizer, model, device)

    # 1. усредняем эмбеддинг в одно число
    data["purpose_mean"] = data["purpose_emb"].apply(lambda x: float(np.mean(x)))

    # 2. выделяем три главные компоненты с предварительным масштабированием и по батчам  
    batch_size = 10_000
    
    if is_train:
        scaler = StandardScaler()
        ipca = IncrementalPCA(n_components=3)

        # обучаем скейлер
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            scaler.partial_fit(batch)

        # обучаем PCA на масштабированных данных
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler.transform(batch)
            ipca.partial_fit(batch_scaled)

    # применяем трансформацию ко всему массиву
    transformed_batches = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к эмбеддингам"):
        batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler.transform(batch)
        transformed_batches.append(ipca.transform(batch_scaled))
        
    purpose_pca_features = np.vstack(transformed_batches)

    # делаем датафрейм
    pca_column_names = [f"purpose_pca_{i+1}" for i in range(3)]
    data[pca_column_names] = purpose_pca_features

    # удалим ненужные столбцы
    data.drop(columns=["purpose", "clean_purpose", "purpose_emb"], inplace=True)

    # генерируем эмбеддинги для названий статей
    # сначала очищаем и лемматизируем тексты
    data["clean_articles__name"] = preprocess_texts_optimized(texts=data["articles__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["articles__name_emb"] = get_embeddings_batch(data["clean_articles__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["articles__name_mean"] = data["articles__name_emb"].apply(lambda x: float(np.mean(x)))
    
    # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_art = StandardScaler()
        ipca_art = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (articles)"):
            batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size])
            scaler_art.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (articles)"):
            batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_art.transform(batch)
            ipca_art.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_art = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к articles embeddings"):
        batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_art.transform(batch)
        transformed_batches_art.append(ipca_art.transform(batch_scaled))

    articles_pca_features = np.vstack(transformed_batches_art)

    # 4) создаём колонки
    art_pca_colnames = [f"articles_pca_{i+1}" for i in range(3)]
    data[art_pca_colnames] = articles_pca_features
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["articles__name", "clean_articles__name", "articles__name_emb"], inplace=True)

    # генерируем эмбеддинги для названий проектов
    # сначала очищаем и лемматизируем тексты
    data["clean_projects__name"] = preprocess_texts_optimized(texts=data["projects__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["projects__name_emb"] = get_embeddings_batch(data["clean_projects__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["projects__name_mean"] = data["projects__name_emb"].apply(lambda x: float(np.mean(x)))
    
    
     # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_pro = StandardScaler()
        ipca_pro = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (projects)"):
            batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size])
            scaler_pro.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (projects)"):
            batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_pro.transform(batch)
            ipca_pro.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_pro = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к projects embeddings"):
        batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_pro.transform(batch)
        transformed_batches_pro.append(ipca_pro.transform(batch_scaled))

    projects_pca_features = np.vstack(transformed_batches_pro)

    # 4) создаём колонки
    pro_pca_colnames = [f"projects_pca_{i+1}" for i in range(3)]
    data[pro_pca_colnames] = projects_pca_features
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["projects__name", "clean_projects__name", "projects__name_emb"], inplace=True)
    
    # генерируем эмбеддинги для названий доноров
    # сначала очищаем и лемматизируем тексты
    data["clean_counterparties__name"] = preprocess_texts_optimized(texts=data["counterparties__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["counterparties__name_emb"] = get_embeddings_batch(data["clean_counterparties__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["counterparties__name_mean"] = data["counterparties__name_emb"].apply(lambda x: float(np.mean(x)))
    
    
     # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_cou = StandardScaler()
        ipca_cou = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (counterparties)"):
            batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size])
            scaler_cou.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (counterparties)"):
            batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_cou.transform(batch)
            ipca_cou.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_cou = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к counterparties embeddings"):
        batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_cou.transform(batch)
        transformed_batches_cou.append(ipca_cou.transform(batch_scaled))

    counterparties_pca_features = np.vstack(transformed_batches_cou)

    # 4) создаём колонки
    cou_pca_colnames = [f"counterparties_pca_{i+1}" for i in range(3)]
    data[cou_pca_colnames] = counterparties_pca_features
    
    
    # удалим ненужные столбцы
    data.drop(columns=["counterparties__name", "clean_counterparties__name", "counterparties__name_emb"], inplace=True)

    
    # сбрасываем неактуальные столбцы
    data = data.drop(columns=[ 
        'date', 'expenditure',
        'payments_amount','user_id','account_id', 
        'contractor_id', 'article_id', 'project_id', 
        'counterpartie_id', 'donor_id', 'donor_cat_id',
        'articles__parent_id', 'projects__parent_id',
        'counterparties__parent_id', 'robots__id'])

    return data,scaler,ipca,scaler_art,ipca_art,scaler_pro,ipca_pro,scaler_cou,ipca_cou


### Загрузим и подготовим данные для обучения моделей

In [7]:
# загружаем новые платежи за вчера
data_full = pd.read_csv("data_download.csv")
'''
url_down = "https://api.lemonpie.tech/api/payments/ai"
headers = {"Authorization": "Bearer YOUR_API_TOKEN"}

params = {
    "limit": 5000,
    "page": 1,
    "expenditure": "incoming"
    #"date_from": "2025-12-04"
}

all_data = []

with tqdm(desc="Загружено страниц", unit=" стр", dynamic_ncols=True) as pbar:
    while True:
        response = requests.get(url_down, headers=headers, params=params)
        if response.status_code != 200:
            print(f"❌ Ошибка загрузки данных с сервера: {response.status_code}")
            break

        result = response.json()
        page_data = result.get("data", [])
        if not page_data:
            break
        
        all_data.extend(page_data)

        params["page"] += 1
        pbar.update(1)
        
# преобразуем в таблицу (вложенные поля будут с __)
data_full = pd.json_normalize(all_data, sep="__")
print(f"Данные загружены с сервера. Количество записей: {len(data_full)}")
data_full.to_csv("data_download.csv", index=False)
'''

'\nurl_down = "https://api.lemonpie.tech/api/payments/ai"\nheaders = {"Authorization": "Bearer YOUR_API_TOKEN"}\n\nparams = {\n    "limit": 5000,\n    "page": 1,\n    "expenditure": "incoming"\n    #"date_from": "2025-12-04"\n}\n\nall_data = []\n\nwith tqdm(desc="Загружено страниц", unit=" стр", dynamic_ncols=True) as pbar:\n    while True:\n        response = requests.get(url_down, headers=headers, params=params)\n        if response.status_code != 200:\n            print(f"❌ Ошибка загрузки данных с сервера: {response.status_code}")\n            break\n\n        result = response.json()\n        page_data = result.get("data", [])\n        if not page_data:\n            break\n        \n        all_data.extend(page_data)\n\n        params["page"] += 1\n        pbar.update(1)\n        \n# преобразуем в таблицу (вложенные поля будут с __)\ndata_full = pd.json_normalize(all_data, sep="__")\nprint(f"Данные загружены с сервера. Количество записей: {len(data_full)}")\ndata_full.to_csv("data

In [8]:
# формируем словарь для разметки тренировочного датасета
main_dict = pd.read_csv("main_dict.csv")
staging_dict = pd.read_csv("staging_dict.csv")

display(main_dict.raw.count(), staging_dict.raw.count())

main_dict = (
    pd.concat([main_dict, staging_dict], ignore_index=True)
      .drop_duplicates(subset=['raw','uc_id'], keep='first')
)
display(main_dict.raw.count())

bad_raw = (
    main_dict
    .groupby('raw')['uc_id']
    .nunique()
    .reset_index(name='uc_id_cnt')
    .query('uc_id_cnt > 1')
) # дубликаты проверим

bad_raw.empty # True - хорошо


813

147

960

True

In [9]:
# отбираем строки, где нет пропусков в признаке, по котрому размечаем метки универсальных категорий
data_full.dropna(subset=['articles__name'], inplace=True)

print(data_full.id.count())

# фильтруем дубликаты и мусорные записи
data_full = data_full.drop_duplicates('id', keep='first')
data_full = data_full[data_full['article_id'] != -1]

data_full.id.count()

660562


660562

In [10]:
data_full['articles__name'] = data_full['articles__name'].str.lower()

data_full['uc__uc_id'] = data_full['articles__name'].map(main_dict.set_index('raw')['uc_id'])

print(data_full.id.count())
print("Количество пропусков:", data_full['uc__uc_id'].isna().sum()) # значит новые статьи относительно словаря
print("Количество статей",data_full['articles__name'].nunique())

data_full['uc__uc_id'].value_counts()


660562
Количество пропусков: 1129
Количество статей 836


1.0    386122
2.0    216544
3.0     17926
4.0     13015
5.0     11022
6.0      9465
7.0      3086
9.0      1135
8.0      1118
Name: uc__uc_id, dtype: int64

In [11]:
# на всякий случай проверим отсутствующие метки во всем датасете, так как в старых платежах могли быть проставлены статьи, а меток нет
# на стадии проверки метрик, мы проверяем метки и статьи у платежей, которые имеют id старше, чем у последнего обучения модели
# если вдруг старые платежи были доразмечены, то их надо добавить в staging для нового обучения 
data_full[data_full['uc__uc_id'].isna()]['articles__name'].value_counts()

статья для удаления    1129
Name: articles__name, dtype: int64

In [12]:
print(data_full.uc__uc_id.isna().sum())

data_full.dropna(subset=['uc__uc_id'], inplace=True)

print(data_full.uc__uc_id.isna().sum())

1129
0


In [13]:
# словарь для наглядности
uc_codex = {"пожертвования от физических лиц (напрямую)":1,
        "пожертвования через платформы":2,
        "пожертвования от юридических лиц (напрямую)":3,
        "прочие недоходные операции":4,
        "продажа услуг":5,
        "продажа товаров":6,
        "финансовые доходы":7,
        "членские и учредительские взносы":8,
        "гранты субсидии конкурсы":9}


In [14]:
# добавим шума в тренировочные данные

delete_aug = nac.RandomCharAug(action="delete", aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)
insert_aug = nac.RandomCharAug(action="insert", aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)
swap_aug   = nac.RandomCharAug(action="swap",   aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)
sub_aug    = nac.RandomCharAug(action="substitute", aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)

augs = [delete_aug, insert_aug, swap_aug, sub_aug]

def corrupt_text(text, n=1):
    if not isinstance(text, str):
        return text
    
    out = text
    for _ in range(n):
        aug = random.choice(augs)
        tmp = aug.augment(out)

        if isinstance(tmp, list):
            tmp = tmp[0]

        out = tmp
    return out

# 1. Отбор 20% строк для всех классов на трейне
df_aug_list = []

for cls in data_full["uc__uc_id"].unique():
    group = data_full[data_full["uc__uc_id"] == cls]
    
    sample = group.sample(
        frac=0.20,
        random_state=RANDOM_STATE
    )
    
    df_aug_list.append(sample)

df_aug = pd.concat(df_aug_list, ignore_index=True)

display(df_aug.shape)
display(df_aug["uc__uc_id"].value_counts())

# 2. Аугментация текстовых столбцов
df_aug['purpose']               = df_aug['purpose'].astype(str).apply(lambda x: corrupt_text(x, n=2))
df_aug['articles__name']        = df_aug['articles__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))
df_aug['projects__name']        = df_aug['projects__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))
df_aug['counterparties__name']  = df_aug['counterparties__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))

# 3. Добавляем шумные данные в train
data_full = pd.concat([data_full, df_aug], ignore_index=True)

display(data_full.shape, data_full.tail(3))

(131886, 31)

1.0    77224
2.0    43309
3.0     3585
4.0     2603
5.0     2204
6.0     1893
7.0      617
9.0      227
8.0      224
Name: uc__uc_id, dtype: int64

(791319, 31)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id
791316,385673,128,40180,2024-05-15,600.00,"БЛГОТВОРИТЕЛЬНЫЙ ВЗН # ОС В ФОPНД ПОМОЩИ ОНКОБОЛЬНЫМ ДЕТЯМ ПО ПРОЕКТУ "" АД + РЕСНАЯ ПОМSОЩ "" (БЕЗ ПУБЛИКАЦИИ) СУМА 2...",8416,incoming,2560,0,0,-1,2012,128.0,195.0,8416.0,195.0,8415.0,пожертвоавние через юр. лиц,2560.0,195.0,561.0,рочная диагностика,2012.0,195.0,2001.0,АДР,NaN,NaN,NaN,3.0
791317,102377,114,10552,2023-09-06,178.00,Благотворительное пожертвование на уствную дiеятльность. НДС не облагаетсdя,47653,incoming,535,0,0,-1,1931,114.0,187.0,47653.0,187.0,0.0,пожертвованя юр лица,535.0,187.0,0.0,Программа Леение,1931.0,187.0,13622.0,Компаини другие,NaN,NaN,NaN,3.0
791318,699966,1027,46,2025-02-07,0.61,"Преевод с карты * 7535, Благотворительноgе пожертвовSание на уставную деятельонVсть. НДС не облагаетяс",33352,incoming,2315,0,0,7410,7389,1027.0,804.0,33352.0,804.0,33347.0,юр. лиц3,2315.0,804.0,0.0,Э_надо_разобрeть,7389.0,804.0,7387.0,? _нужно_проставить_дKонора,7410.0,804.0,NaN,3.0


In [15]:
# делаем полную подготовку данных с кодированием текстов

data_train_prepared_path = 'data_train_prepared.parquet'
scaler_emb_path = 'scaler.pkl'
ipca_emb_path = 'ipca.pkl'
scaler_art_emb_path = 'scaler_art.pkl'
ipca_art_emb_path = 'ipca_art.pkl'
scaler_pro_emb_path = 'scaler_pro.pkl'
ipca_pro_emb_path = 'ipca_pro.pkl'
scaler_cou_emb_path = 'scaler_cou.pkl'
ipca_cou_emb_path = 'ipca_cou.pkl'


if os.path.exists(data_train_prepared_path) and os.path.exists(scaler_emb_path) and os.path.exists(ipca_emb_path):
    data_train_prepared = pd.read_parquet(data_train_prepared_path)
    scaler = joblib.load(scaler_emb_path)
    ipca = joblib.load(ipca_emb_path)
    scaler_art = joblib.load(scaler_art_emb_path)
    ipca_art = joblib.load(ipca_art_emb_path)
    scaler_pro = joblib.load(scaler_pro_emb_path)
    ipca_pro = joblib.load(ipca_pro_emb_path)
    scaler_cou = joblib.load(scaler_cou_emb_path)
    ipca_cou = joblib.load(ipca_cou_emb_path)

else:
    data_train_prepared,scaler,ipca,scaler_art,ipca_art,scaler_pro,ipca_pro,scaler_cou,ipca_cou = prepare_data(data_full,is_train=True,scaler=None,ipca=None,
                                                                          scaler_art=None, ipca_art=None,
                                                                          scaler_pro=None, ipca_pro=None,
                                                                          scaler_cou=None, ipca_cou=None)

    data_train_prepared.to_parquet(data_train_prepared_path)
    joblib.dump(scaler, scaler_emb_path)
    joblib.dump(ipca, ipca_emb_path)
    joblib.dump(scaler_art, scaler_art_emb_path)
    joblib.dump(ipca_art, ipca_art_emb_path)
    joblib.dump(scaler_pro, scaler_pro_emb_path)
    joblib.dump(ipca_pro, ipca_pro_emb_path)
    joblib.dump(scaler_cou, scaler_cou_emb_path)
    joblib.dump(ipca_cou, ipca_cou_emb_path)

🔍 Запуск предобработки для 791319 текстов...


Очистка: 100%|██████████| 791319/791319 [00:02<00:00, 268244.05it/s]


🔍 Загрузка spaCy модели: 'ru_core_news_sm'.
🔍 Лемматизация будет выполнена в 7 потоках.


Лемматизация (CPU): 100%|██████████| 791319/791319 [19:34<00:00, 673.87it/s] 


🔍 Предобработка завершена. Обработано 791319 текстов.


...

Лемматизация (CPU): 100%|██████████| 791319/791319 [08:30<00:00, 1550.72it/s]


🔍 Предобработка завершена. Обработано 791319 текстов.
🔍 Начало генерации эмбеддингов для 791319 текстов на устройстве 'cuda'.


Генерация эмбеддингов: 100%|██████████| 12365/12365 [03:09<00:00, 65.35it/s]


🔍 Генерация эмбеддингов завершена


Применяем PCA к counterparties embeddings: 100%|██████████| 80/80 [00:08<00:00,  9.85it/s]


In [16]:
import spacy
print(spacy.__version__)
print(spacy.__file__)

3.6.1
/home/jupyter/.local/lib/python3.10/site-packages/spacy/__init__.py


In [17]:
data_train_prepared.dropna(subset=['uc__uc_id'], inplace=True)

In [18]:
data_train_prepared.uc__uc_id.isna().sum()

0

In [19]:
# разделим признаки
X_train = data_train_prepared.drop(columns=['id','uc__uc_id'])
y_train = data_train_prepared['uc__uc_id']

#### Обучаем модель catboost

In [20]:
X_train.columns

Index(['purpose_mean', 'purpose_pca_1', 'purpose_pca_2', 'purpose_pca_3',
       'articles__name_mean', 'articles_pca_1', 'articles_pca_2',
       'articles_pca_3', 'projects__name_mean', 'projects_pca_1',
       'projects_pca_2', 'projects_pca_3', 'counterparties__name_mean',
       'counterparties_pca_1', 'counterparties_pca_2', 'counterparties_pca_3'],
      dtype='object')

In [21]:

if utils.get_gpu_device_count() > 0:
    task_type = "GPU"
else:
    task_type = "CPU"

#cat_features = ["day_of_week", "month", "is_weekend"]
cat_features = []
    
# создаем модель
model_cb = CatBoostClassifier(
                        #silent=True,
                        random_seed=RANDOM_STATE, 
                        cat_features=cat_features,
                        task_type=task_type,
                        thread_count=-1,
                        verbose = 100
)

# сетка гиперпараметров
param_grid = [
    # CatBoostClassifier
    {
        "iterations": [1000],
        "depth": [6],
        "learning_rate": [0.05],
        "l2_leaf_reg": [9],
        "loss_function": ["MultiClassOneVsAll"], #MultiClassOneVsAll, MultiClass
        "max_bin": [256],
        "random_strength": [5],
        "min_data_in_leaf":[1],
        "auto_class_weights":['SqrtBalanced']
    }
]

# полный перебор гиперпараметров с помощью GridSearchCV
grid_search_cb = GridSearchCV(
    model_cb,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=2),
    scoring="f1_weighted",
    n_jobs=1,
    refit=True
    # error_score=np.nan
)

# обучение модели
grid_search_cb.fit(X_train, y_train)

# выбираем лучшую модель и сохраняем 
best_model_cb = grid_search_cb.best_estimator_    
best_model_cb.save_model("model_cb.cbm")
print('Параметры лучшей модели: ', best_model_cb.get_params())


0:	learn: 0.6536661	total: 309ms	remaining: 5m 8s
100:	learn: 0.0901191	total: 1.08s	remaining: 9.64s
200:	learn: 0.0608395	total: 1.83s	remaining: 7.28s
300:	learn: 0.0490440	total: 2.56s	remaining: 5.95s
400:	learn: 0.0421291	total: 3.29s	remaining: 4.91s
500:	learn: 0.0374122	total: 4.01s	remaining: 3.99s
600:	learn: 0.0341117	total: 4.75s	remaining: 3.15s
700:	learn: 0.0314322	total: 5.47s	remaining: 2.33s
800:	learn: 0.0292300	total: 6.22s	remaining: 1.55s
900:	learn: 0.0274526	total: 6.98s	remaining: 767ms
999:	learn: 0.0260259	total: 7.71s	remaining: 0us
0:	learn: 0.6512105	total: 7.98ms	remaining: 7.97s
100:	learn: 0.0397654	total: 713ms	remaining: 6.34s
200:	learn: 0.0167354	total: 1.41s	remaining: 5.59s
300:	learn: 0.0099776	total: 2.11s	remaining: 4.91s
400:	learn: 0.0071434	total: 2.77s	remaining: 4.14s
500:	learn: 0.0054968	total: 3.45s	remaining: 3.43s
600:	learn: 0.0044535	total: 4.1s	remaining: 2.72s
700:	learn: 0.0037255	total: 4.77s	remaining: 2.04s
800:	learn: 0.0031

In [22]:
# проверяем важности
importances = best_model_cb.get_feature_importance()

feat_imp = pd.DataFrame({
    'feature': best_model_cb.feature_names_,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(20))

                      feature  importance
5              articles_pca_1   16.692194
6              articles_pca_2   13.078999
7              articles_pca_3   12.161956
13       counterparties_pca_1    9.205645
4         articles__name_mean    8.907858
15       counterparties_pca_3    5.150310
14       counterparties_pca_2    4.933974
2               purpose_pca_2    4.774398
11             projects_pca_3    4.456784
3               purpose_pca_3    4.097399
1               purpose_pca_1    4.069110
9              projects_pca_1    3.797343
10             projects_pca_2    3.706348
12  counterparties__name_mean    2.871232
0                purpose_mean    1.082071
8         projects__name_mean    1.014380


In [23]:
# пересохраняем словари
main_dict.to_csv("main_dict.csv")

staging_dict = staging_dict.head(0)
staging_dict.to_csv('staging_dict.csv')


In [24]:
# фиксируем id и дату последнего платежа из текущего обучения
train_history = pd.read_csv('train_history.csv')

last_row = data_full.loc[data_full.id == data_full['id'].max()].iloc[0]

new_row = {
    'run_date': pd.Timestamp.today().date(),
    'last_payment_date': pd.to_datetime(last_row['date']).date(),
    'last_payment_id': int(last_row['id'])
}

train_history = pd.concat(
    [train_history, pd.DataFrame([new_row])],
    ignore_index=True
)

train_history.drop_duplicates(
    subset=['last_payment_id'],
    keep='last',
    inplace=True
)

train_history.to_csv('train_history.csv', index=False)